# 12 — Ax Bayesian optimization for rollout α (CTL-03)

**Decision:** ADR 0060 / CTL-03=B — tune the demand fractile `alpha` for the **rollout** ladder arm by maximizing closed-loop **episode profit** (SIM-01=B), using [Ax](https://ax.dev/) instead of a fixed grid.

Each Ax trial evaluates one candidate α on **K stochastic demand realizations** (distinct `root_seed`s). We report `(mean, sem)` to Ax so observation noise is explicit (see Ax trial-evaluation docs).

Scoring uses `evaluate_alpha_episode_profit("rollout", ...)` in `sim/alpha_tune.py` — **Rust-first** when `blueberries_voi._core` is built (`maturin develop`).

Companion: nb 11 grid-searches constant / rung0 / sw / rollout. This notebook focuses on **BO for rollout only**.

**Defaults are smoke-sized.** Set `FULL_RUN = True` for desktop budgets (slow).

## Setup

From the repo root:

```bash
uv sync --extra notebooks --extra viz --extra rust
uv run maturin develop --manifest-path crates/voi_py/Cargo.toml
uv run jupyter lab
```

First `uv sync` with `ax-platform` may take several minutes (PyTorch + BoTorch).

Add `--extra data` if you set `USE_ABDELLA = True` (Parquet shipments).

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from tqdm.auto import tqdm

from blueberries_voi.backend import rust_available, rust_core
from blueberries_voi.sim.alpha_tune import (
    DEFAULT_CI_ALPHAS,
    DEFAULT_DESKTOP_ALPHAS,
    evaluate_alpha_episode_profit,
    tune_alpha_grid,
)
from blueberries_voi.sim.bakeoff_rollout import DEFAULT_ROLLOUT_H
from blueberries_voi.sim.profit import DEFAULT_PROFIT_COSTS
from blueberries_voi.sim.shipments import default_shipments, smoke_cool_shipments

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "blueberries_voi").is_dir():
    REPO_ROOT = REPO_ROOT.parent

ARM = "rollout"
FULL_RUN = False
USE_ABDELLA = False
ALPHA_BOUNDS = (0.5, 0.95)

if FULL_RUN:
    N_BURN, N_SCORE = 28, 28
    ROLLOUT_H = int(DEFAULT_ROLLOUT_H)
    N_ROLLOUT_PATHS = 8
    CANDIDATE_CASE_RADIUS = 2
    K_BO_SEEDS = 6
    N_AX_TRIALS = 20
    K_VAL_SEEDS = 5
    GRID_ALPHAS = DEFAULT_DESKTOP_ALPHAS
else:
    N_BURN, N_SCORE = 2, 5
    ROLLOUT_H = 7
    N_ROLLOUT_PATHS = 2
    CANDIDATE_CASE_RADIUS = 2
    K_BO_SEEDS = 4
    N_AX_TRIALS = 10
    K_VAL_SEEDS = 3
    GRID_ALPHAS = tuple(DEFAULT_CI_ALPHAS)

RNG = np.random.default_rng(20260817)
BO_SEEDS = [int(RNG.integers(0, 2**31 - 1)) for _ in range(K_BO_SEEDS)]
VAL_SEEDS = [int(RNG.integers(0, 2**31 - 1)) for _ in range(K_VAL_SEEDS)]

shipments = default_shipments() if USE_ABDELLA else smoke_cool_shipments()
costs = DEFAULT_PROFIT_COSTS
OUTPUT_JSON = REPO_ROOT / "outputs" / "rollout_alpha_bo.json"

rust_fn = getattr(rust_core, "evaluate_alpha_tune_episode_py", None) if rust_core else None
print(f"Rust kernel: {rust_available() and rust_fn is not None}")
print(f"α bounds: {ALPHA_BOUNDS}")
print(f"episode: n_burn={N_BURN}, n_score={N_SCORE}")
print(
    f"rollout: H={ROLLOUT_H}, n_paths={N_ROLLOUT_PATHS}, "
    f"radius={CANDIDATE_CASE_RADIUS}"
)
print(f"BO seeds (K={K_BO_SEEDS}): {BO_SEEDS}")
print(f"validation seeds: {VAL_SEEDS}")
print(f"Ax trials: {N_AX_TRIALS}")

%matplotlib inline
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": 0.3})

## Objective: rollout episode profit with demand replicates

One Ax observation per α = mean and SEM of K episode profits at fixed `BO_SEEDS`. SEM uses sample std / √K (standard error of the mean).

In [ ]:
def evaluate_rollout_profit(alpha: float, root_seed: int) -> float:
    return float(
        evaluate_alpha_episode_profit(
            ARM,
            float(alpha),
            int(root_seed),
            shipments=shipments,
            costs=costs,
            n_burn=N_BURN,
            n_score=N_SCORE,
            rollout_h=ROLLOUT_H,
            n_rollout_paths=N_ROLLOUT_PATHS,
            candidate_case_radius=CANDIDATE_CASE_RADIUS,
        )
    )


def evaluate_with_replicates(alpha: float, seeds: list[int]) -> tuple[float, float]:
    profits = [evaluate_rollout_profit(alpha, s) for s in seeds]
    arr = np.asarray(profits, dtype=float)
    mean = float(arr.mean())
    sem = float(arr.std(ddof=1) / np.sqrt(len(arr))) if len(arr) > 1 else 0.0
    return mean, sem


demo_mean, demo_sem = evaluate_with_replicates(0.9, BO_SEEDS[:2])
print(f"smoke α=0.9 on 2 seeds: mean={demo_mean:.2f}, sem={demo_sem:.3f}")

## Textbook fractile reference (CTL-03 context)

In [ ]:
alpha_theory_penalty = costs.stockout_penalty / (
    costs.stockout_penalty + costs.waste_cost
)
alpha_theory_margin = costs.unit_margin / (costs.unit_margin + costs.waste_cost)
print(f"Textbook (penalty / waste): {alpha_theory_penalty:.3f}")
print(f"Textbook (margin / waste):  {alpha_theory_margin:.3f}")

## Ax Bayesian optimization loop

Uses the modern [`ax.api.Client`](https://ax.dev/docs/tutorials/getting_started/) (replaces deprecated `AxClient`). Maximize `episode_profit`. Each completed trial passes `(mean, sem)` so Ax models heteroskedastic observation noise.

In [ ]:
ax_client = Client()
ax_client.configure_experiment(
    parameters=[
        RangeParameterConfig(
            name="alpha",
            parameter_type="float",
            bounds=ALPHA_BOUNDS,
        ),
    ],
)
ax_client.configure_optimization(objective="episode_profit")

trial_log: list[dict[str, Any]] = []
for _ in tqdm(range(N_AX_TRIALS), desc="Ax trials"):
    trials = ax_client.get_next_trials(max_trials=1)
    trial_index, parameters = next(iter(trials.items()))
    alpha = float(parameters["alpha"])
    mean, sem = evaluate_with_replicates(alpha, BO_SEEDS)
    ax_client.complete_trial(
        trial_index=trial_index,
        raw_data={"episode_profit": (mean, sem)},
    )
    trial_log.append(
        {
            "trial_index": int(trial_index),
            "alpha": alpha,
            "mean_profit": mean,
            "sem": sem,
        }
    )

best_parameters, _prediction, best_index, _name = ax_client.get_best_parameterization()
best_alpha_bo = float(best_parameters["alpha"])
print(f"Ax best α (model-predicted): {best_alpha_bo:.4f} (trial {best_index})")

## Grid baseline on the same BO seed panel

For each grid α, use the same replicate-mean objective (not `tune_alpha_grid`'s single-seed CRN).

In [ ]:
grid_means: list[float] = []
grid_sems: list[float] = []
for a in tqdm(GRID_ALPHAS, desc="grid baseline"):
    m, s = evaluate_with_replicates(float(a), BO_SEEDS)
    grid_means.append(m)
    grid_sems.append(s)

best_grid_idx = int(np.argmax(grid_means))
best_alpha_grid = float(GRID_ALPHAS[best_grid_idx])
print(f"Grid best α (max replicate mean): {best_alpha_grid:.4f}")

# Single-seed tune_alpha_grid (CRN across alphas on one seed) for comparison
crn_seed = BO_SEEDS[0]
best_alpha_crn = tune_alpha_grid(
    ARM,
    alphas=GRID_ALPHAS,
    root_seed=crn_seed,
    shipments=shipments,
    costs=costs,
    n_burn=N_BURN,
    n_score=N_SCORE,
    rollout_h=ROLLOUT_H,
    n_rollout_paths=N_ROLLOUT_PATHS,
    candidate_case_radius=CANDIDATE_CASE_RADIUS,
)
print(f"tune_alpha_grid on seed {crn_seed}: α*={best_alpha_crn:.4f}")

## Held-out validation

Score BO and grid winners on seeds **not** used during optimization.

In [ ]:
def validation_mean(alpha: float) -> float:
    return float(np.mean([evaluate_rollout_profit(alpha, s) for s in VAL_SEEDS]))


val_bo = validation_mean(best_alpha_bo)
val_grid = validation_mean(best_alpha_grid)
print(f"Validation mean profit — Ax α={best_alpha_bo:.4f}: {val_bo:.2f}")
print(f"Validation mean profit — grid α={best_alpha_grid:.4f}: {val_grid:.2f}")

## Diagnostics

In [ ]:
alphas_ax = [t["alpha"] for t in trial_log]
means_ax = [t["mean_profit"] for t in trial_log]
sems_ax = [t["sem"] for t in trial_log]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
ax.errorbar(alphas_ax, means_ax, yerr=sems_ax, fmt="o", alpha=0.7, label="Ax trials")
ax.plot(GRID_ALPHAS, grid_means, "s--", color="#2563eb", label="grid baseline")
ax.axvline(best_alpha_bo, color="#16a34a", ls="--", label=f"Ax α*={best_alpha_bo:.2f}")
ax.axvline(best_alpha_grid, color="#9333ea", ls=":", label=f"grid α*={best_alpha_grid:.2f}")
ax.axvline(alpha_theory_penalty, color="#dc2626", ls=":", alpha=0.7, label="theory (penalty)")
ax.set_xlabel("α")
ax.set_ylabel("Episode profit (replicate mean)")
ax.set_title(f"Rollout α — {ARM} (K={K_BO_SEEDS} BO seeds)")
ax.legend(fontsize=7, loc="best")

ax = axes[1]
order = np.argsort(alphas_ax)
ax.plot(np.arange(len(trial_log)), np.array(means_ax)[order], "o-")
ax.set_xlabel("Ax trial (sorted by α)")
ax.set_ylabel("Replicate mean profit")
ax.set_title("BO exploration")

fig.tight_layout()
plt.show()

## Save results (optional)

Writes to `outputs/` (gitignored) — not `experiments/tuned_alpha.json`.

In [ ]:
payload: dict[str, Any] = {
    "arm": ARM,
    "full_run": FULL_RUN,
    "rust_kernel": bool(rust_available() and rust_fn is not None),
    "alpha_bounds": list(ALPHA_BOUNDS),
    "n_burn": N_BURN,
    "n_score": N_SCORE,
    "rollout_h": ROLLOUT_H,
    "n_rollout_paths": N_ROLLOUT_PATHS,
    "candidate_case_radius": CANDIDATE_CASE_RADIUS,
    "bo_seeds": BO_SEEDS,
    "val_seeds": VAL_SEEDS,
    "best_alpha_ax": best_alpha_bo,
    "best_alpha_grid": best_alpha_grid,
    "best_alpha_tune_alpha_grid_crn": float(best_alpha_crn),
    "validation_mean_ax": val_bo,
    "validation_mean_grid": val_grid,
    "trials": trial_log,
}
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
print(f"Wrote {OUTPUT_JSON}")

## Takeaways

1. **Rollout α** is tuned by maximizing closed-loop episode profit, not the textbook newsvendor fractile.
2. **Ax** (`ax.api.Client`) receives `(mean, sem)` over K demand seeds per α — no explicit demand model required.
3. **Rust path** — `evaluate_alpha_episode_profit("rollout", ...)` uses `voi_core` when `_core` is built.
4. **Validation** on held-out seeds guards against overfitting a small BO panel.
5. Scale up with `FULL_RUN = True` for production-style rollout budgets (H=28, 8 paths).